# NYC Taxi Trip Duration Prediction

## Project Overview

This notebook builds a machine learning regression system to predict NYC Yellow Taxi trip duration using feature-engineered trip data.

The target variable for prediction is:

**`log_trip_duration`**

The workflow includes:

- Loading processed feature data
- Preparing training and testing datasets
- Creating preprocessing pipelines
- Training and evaluating multiple regression models
- Selecting the best performing model
- Saving the final champion model for inference

---

## Models Evaluated

The following regression models were evaluated:

### Linear Models
- Linear Regression
- Tuned Ridge Regression
- Tuned Lasso Regression
- Tuned Elastic Net Regression

### Tree-Based Models
- Tuned Decision Tree Regressor
- Tuned Random Forest Regressor
- Tuned Gradient Boosting Regressor

---

## Evaluation Metrics

The models are compared using:

| Metric | Purpose |
|---|---|
| MAE | Measures average prediction error |
| RMSE | Primary metric for model ranking |
| Median AE | Robust error measurement |
| R² Score | Measures explained variance |
| Prediction Time | Measures inference speed |

The final model will be selected based on the best RMSE performance while maintaining strong generalization.

In [78]:
# =============================================================================
# FIX PYTHON PATH
# =============================================================================

import sys
from pathlib import Path


PROJECT_ROOT = Path(
    r"C:\Users\abuba\Downloads\ML-Projects\Day-5"
)


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


print("Project added to Python path")

Project added to Python path


In [67]:
# =============================================================================
# BLOCK 0: Project Setup
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import joblib


PROJECT_ROOT = Path(
    r"C:\Users\abuba\Downloads\ML-Projects\Day-5"
)


DATA_DIR = PROJECT_ROOT / "data"

PROCESSED_DIR = DATA_DIR / "processed"

MODELS_DIR = PROJECT_ROOT / "models"

LINEAR_DIR = MODELS_DIR / "linear"

TREE_DIR = MODELS_DIR / "tree"


print("=" * 70)
print("Project Paths Ready")
print("=" * 70)

print(PROJECT_ROOT)

Project Paths Ready
C:\Users\abuba\Downloads\ML-Projects\Day-5


In [68]:
# =============================================================================
# BLOCK 1: Load Feature Dataset
# =============================================================================

FEATURE_PATH = (
    PROCESSED_DIR
    / "featured_data.parquet"
)


featured_df = pd.read_parquet(
    FEATURE_PATH
)


print("=" * 70)
print("Feature Dataset Loaded")
print("=" * 70)

print("Rows    :", featured_df.shape[0])
print("Columns :", featured_df.shape[1])

Feature Dataset Loaded
Rows    : 18378485
Columns : 32


In [69]:
# =============================================================================
# BLOCK 2: Prepare X and y
# =============================================================================

TARGET = "log_trip_duration"


DROP_COLUMNS = [
    "log_trip_duration",
    "trip_duration_minutes",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
]


X = featured_df.drop(
    columns=DROP_COLUMNS
)


y = featured_df[TARGET]


print("=" * 70)
print("Dataset Prepared")
print("=" * 70)

print("X shape:", X.shape)
print("y shape:", y.shape)


print("\nColumns:")
print(X.columns.tolist())

Dataset Prepared
X shape: (18378485, 28)
y shape: (18378485,)

Columns:
['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'congestion_surcharge', 'airport_fee', 'pickup_year', 'pickup_month', 'pickup_day', 'pickup_hour', 'pickup_dayofweek', 'pickup_week', 'pickup_weekend', 'pickup_quarter', 'rush_hour', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'same_borough', 'same_location', 'is_airport_trip', 'distance_squared', 'log_trip_distance', 'distance_per_passenger']


In [70]:
# =============================================================================
# BLOCK 3: Check Feature Types
# =============================================================================

numeric_features = X.select_dtypes(
    include=[
        "int8",
        "int16",
        "int32",
        "int64",
        "float32",
        "float64"
    ]
).columns.tolist()


categorical_features = X.select_dtypes(
    include=[
        "object",
        "string",
        "category"
    ]
).columns.tolist()


print("=" * 70)
print("Feature Check")
print("=" * 70)

print("Total Features      :", X.shape[1])
print("Numeric Features    :", len(numeric_features))
print("Categorical Features:", len(categorical_features))

Feature Check
Total Features      : 28
Numeric Features    : 27
Categorical Features: 1


In [71]:
# =============================================================================
# BLOCK 4: Train Test Split
# =============================================================================

from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


print("=" * 70)
print("Split Completed")
print("=" * 70)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

Split Completed
X_train: (14702788, 28)
X_test : (3675697, 28)
y_test : (3675697,)


In [72]:
# =============================================================================
# BLOCK 5: Create Preprocessors
# =============================================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


# -----------------------------
# Linear Preprocessor
# -----------------------------

linear_preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",

            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        )
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    )
                ]
            ),

            numeric_features
        ),


        (
            "cat",

            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore"
                        )
                    )
                ]
            ),

            categorical_features
        )
    ]
)



# -----------------------------
# Tree Preprocessor
# -----------------------------

tree_preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",

            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        )
                    )
                ]
            ),

            numeric_features
        ),


        (
            "cat",

            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore"
                        )
                    )
                ]
            ),

            categorical_features
        )
    ]
)


print("Preprocessors Created")

Preprocessors Created


In [73]:
# =============================================================================
# BLOCK 6: Transform Train/Test
# =============================================================================


X_train_linear = linear_preprocessor.fit_transform(
    X_train
)

X_test_linear = linear_preprocessor.transform(
    X_test
)



X_train_tree = tree_preprocessor.fit_transform(
    X_train
)

X_test_tree = tree_preprocessor.transform(
    X_test
)



print("=" * 70)
print("Transformation Complete")
print("=" * 70)

print("Linear Train:", X_train_linear.shape)
print("Linear Test :", X_test_linear.shape)

print("Tree Train:", X_train_tree.shape)
print("Tree Test :", X_test_tree.shape)

Transformation Complete
Linear Train: (14702788, 29)
Linear Test : (3675697, 29)
Tree Train: (14702788, 29)
Tree Test : (3675697, 29)


## Load Linear + Tree Models

In [ ]:
import joblib


# -----------------------------
# Linear Models
# -----------------------------

linear_regression = joblib.load(
    LINEAR_DIR / "linear_regression.joblib"
)

ridge_model = joblib.load(
    LINEAR_DIR / "tuned_ridge.joblib"
)

lasso_model = joblib.load(
    LINEAR_DIR / "tuned_lasso.joblib"
)

elastic_net_model = joblib.load(
    LINEAR_DIR / "tuned_elastic_net.joblib"
)



# -----------------------------
# Tree Models
# -----------------------------

gradient_boosting = joblib.load(
    TREE_DIR / "tuned_gradient_boosting.joblib"
)

random_forest = joblib.load(
    TREE_DIR / "tuned_random_forest.joblib"
)

decision_tree = joblib.load(
    TREE_DIR / "tuned_decision_tree.joblib"
)



print("=" * 70)
print("All Models Loaded")
print("=" * 70)


print("Linear Regression:",
      linear_regression.n_features_in_)

print("Gradient Boosting:",
      gradient_boosting.n_features_in_)

print("Random Forest:",
      random_forest.n_features_in_)

print("Decision Tree:",
      decision_tree.n_features_in_)

All Models Loaded
Linear Regression: 29
Gradient Boosting: 29
Random Forest: 29
Decision Tree: 29


In [79]:
# =============================================================================
# BLOCK 8: Import Evaluation Utilities
# =============================================================================

from src.models.evaluate import (
    evaluate_model,
    create_results_dataframe
)


print("Evaluation utilities loaded")

Evaluation utilities loaded


In [80]:
# =============================================================================
# BLOCK 9: Linear Model Evaluation
# =============================================================================

linear_results = []


linear_results.append(
    evaluate_model(
        linear_regression,
        X_test_linear,
        y_test,
        model_name="Linear Regression"
    )
)


linear_results.append(
    evaluate_model(
        ridge_model,
        X_test_linear,
        y_test,
        model_name="Tuned Ridge"
    )
)


linear_results.append(
    evaluate_model(
        lasso_model,
        X_test_linear,
        y_test,
        model_name="Tuned Lasso"
    )
)


linear_results.append(
    evaluate_model(
        elastic_net_model,
        X_test_linear,
        y_test,
        model_name="Tuned Elastic Net"
    )
)


linear_results_df = create_results_dataframe(
    linear_results
)


linear_results_df

,Model,MAE,RMSE,Median AE,R²,Training Time (s),Prediction Time (s)
0,Linear Regression,1.067203,1.139193,1.092329,-1.884348,0.0,0.871923
1,Tuned Elastic Net,1.460051,1.527921,1.435995,-4.188655,0.0,0.176894
2,Tuned Ridge,1.468128,1.537644,1.443836,-4.254905,0.0,0.187314
3,Tuned Lasso,1.468881,1.538098,1.444640,-4.258004,0.0,0.181160


In [81]:
# =============================================================================
# BLOCK 10: Tree Model Evaluation
# =============================================================================

tree_results = []


tree_results.append(
    evaluate_model(
        gradient_boosting,
        X_test_tree,
        y_test,
        model_name="Tuned Gradient Boosting"
    )
)


tree_results.append(
    evaluate_model(
        random_forest,
        X_test_tree,
        y_test,
        model_name="Tuned Random Forest"
    )
)


tree_results.append(
    evaluate_model(
        decision_tree,
        X_test_tree,
        y_test,
        model_name="Tuned Decision Tree"
    )
)


tree_results_df = create_results_dataframe(
    tree_results
)


print("=" * 70)
print("Tree Models Evaluation")
print("=" * 70)


tree_results_df

Tree Models Evaluation


,Model,MAE,RMSE,Median AE,R²,Training Time (s),Prediction Time (s)
0,Tuned Random Forest,0.195348,0.261017,0.154788,0.848577,0.0,23.773557
1,Tuned Gradient Boosting,0.201782,0.266050,0.161919,0.842681,0.0,4.784494
2,Tuned Decision Tree,0.232723,0.308348,0.184842,0.788682,0.0,0.905185


In [82]:
# =============================================================================
# BLOCK 11: Combine All Model Results
# =============================================================================

all_results = (
    linear_results
    +
    tree_results
)


final_results_df = create_results_dataframe(
    all_results
)


print("=" * 70)
print("FINAL MODEL LEADERBOARD")
print("=" * 70)


final_results_df

FINAL MODEL LEADERBOARD


,Model,MAE,RMSE,Median AE,R²,Training Time (s),Prediction Time (s)
0,Tuned Random Forest,0.195348,0.261017,0.154788,0.848577,0.0,23.773557
1,Tuned Gradient Boosting,0.201782,0.266050,0.161919,0.842681,0.0,4.784494
2,Tuned Decision Tree,0.232723,0.308348,0.184842,0.788682,0.0,0.905185
3,Linear Regression,1.067203,1.139193,1.092329,-1.884348,0.0,0.871923
4,Tuned Elastic Net,1.460051,1.527921,1.435995,-4.188655,0.0,0.176894
5,Tuned Ridge,1.468128,1.537644,1.443836,-4.254905,0.0,0.187314
6,Tuned Lasso,1.468881,1.538098,1.444640,-4.258004,0.0,0.181160


In [83]:
# =============================================================================
# BLOCK 12: Save Champion Model
# =============================================================================

from pathlib import Path
import joblib


# Create champion folder

CHAMP_DIR = PROJECT_ROOT / "models" / "champ"


CHAMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Save champion model

champion_model_path = (
    CHAMP_DIR
    / "champion_random_forest.joblib"
)


joblib.dump(
    random_forest,
    champion_model_path
)


print("=" * 70)
print("Champion Model Saved")
print("=" * 70)

print(
    champion_model_path
)

Champion Model Saved
C:\Users\abuba\Downloads\ML-Projects\Day-5\models\champ\champion_random_forest.joblib


# Final Model Selection

## Champion Model

The selected production model is:

# Tuned Random Forest Regressor

The model achieved the best overall performance among all evaluated models.

---

## Final Performance

| Metric | Score |
|---|---:|
| MAE | 0.195348 |
| RMSE | 0.261017 |
| Median AE | 0.154788 |
| R² Score | 0.848577 |

---

## Saved Model Artifacts

The final production files are stored in:
